# Defensive and Tactical Features

**Chapter 8: Feature Engineering for Soccer Predictions**

## Learning Objectives

- Build defensive metrics (tackles, interceptions, recoveries)
- Calculate PPDA (Passes Per Defensive Action) for pressing intensity
- Create set piece features
- Engineer contextual features (rest days, fixture congestion)
- Build momentum and streak features
- Understand different defensive styles through data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("Defensive Analytics Toolkit Ready!")

## Generate Simulated Match Data

We'll create comprehensive match data including defensive actions.

In [ ]:
# Generate simulated season data with defensive metrics
np.random.seed(42)

teams = ['Arsenal', 'Chelsea', 'Liverpool', 'Man City', 'Man United', 'Tottenham',
         'Leicester', 'West Ham', 'Everton', 'Aston Villa', 'Newcastle', 'Brighton',
         'Crystal Palace', 'Wolves', 'Southampton', 'Burnley', 'Leeds', 'Watford',
         'Norwich', 'Brentford']

matches = []
match_id = 1
start_date = datetime(2023, 8, 12)

for round_num in range(2):  # Home and away
    for i, home_team in enumerate(teams):
        for j, away_team in enumerate(teams):
            if i != j:
                match_date = start_date + timedelta(days=match_id * 4)
                
                # Basic stats
                home_score = np.random.poisson(1.5)
                away_score = np.random.poisson(1.0)
                
                # Defensive actions
                home_tackles = np.random.poisson(15)
                away_tackles = np.random.poisson(15)
                home_interceptions = np.random.poisson(10)
                away_interceptions = np.random.poisson(10)
                home_recoveries = np.random.poisson(40)
                away_recoveries = np.random.poisson(40)
                
                # Opponent passes (for PPDA calculation)
                home_opponent_passes = np.random.poisson(350)  # Away team passes
                away_opponent_passes = np.random.poisson(350)  # Home team passes
                
                # Set pieces
                home_corners = np.random.poisson(5)
                away_corners = np.random.poisson(4)
                home_free_kicks = np.random.poisson(15)
                away_free_kicks = np.random.poisson(15)
                
                matches.append({
                    'match_id': match_id,
                    'match_date': match_date,
                    'home_team': home_team,
                    'away_team': away_team,
                    'home_score': home_score,
                    'away_score': away_score,
                    'home_tackles': home_tackles,
                    'away_tackles': away_tackles,
                    'home_interceptions': home_interceptions,
                    'away_interceptions': away_interceptions,
                    'home_recoveries': home_recoveries,
                    'away_recoveries': away_recoveries,
                    'home_opponent_passes': home_opponent_passes,
                    'away_opponent_passes': away_opponent_passes,
                    'home_corners': home_corners,
                    'away_corners': away_corners,
                    'home_free_kicks': home_free_kicks,
                    'away_free_kicks': away_free_kicks
                })
                
                match_id += 1

matches_df = pd.DataFrame(matches)
matches_df = matches_df.sort_values('match_date').reset_index(drop=True)

print(f"Generated {len(matches_df)} matches")
print(f"\nSample data:")
matches_df.head()

---

# Defensive Metrics: The Other Side of the Coin

> **"Offense wins games, but defense wins championships."**

Yet defensive metrics often get less attention than attacking ones. Let's fix that. A solid defense is about more than just preventing goals—it's about winning the ball back, disrupting the opponent's play, and starting your own attacks.

## Tackles, Interceptions, and Recoveries

Defensive actions come in several flavors:

- **Tackles**: Direct challenges for the ball
- **Interceptions**: Reading the play and cutting out a pass
- **Recoveries**: Regaining possession of a loose ball

Each tells us something different about a team's defensive style.

In [ ]:
# Calculate total defensive actions
matches_df['home_defensive_actions'] = (matches_df['home_tackles'] + 
                                         matches_df['home_interceptions'] + 
                                         matches_df['home_recoveries'])

matches_df['away_defensive_actions'] = (matches_df['away_tackles'] + 
                                         matches_df['away_interceptions'] + 
                                         matches_df['away_recoveries'])

print("Defensive actions calculated!")
print("\nSample:")
print(matches_df[['home_team', 'away_team', 'home_tackles', 'home_interceptions', 
                  'home_recoveries', 'home_defensive_actions']].head())

In [ ]:
# Create long-form for defensive analysis
defensive_data = pd.concat([
    matches_df[['match_date', 'home_team', 'home_tackles', 'home_interceptions', 
                'home_recoveries', 'home_defensive_actions']].rename(
        columns={'home_team': 'team', 'home_tackles': 'tackles', 
                'home_interceptions': 'interceptions', 'home_recoveries': 'recoveries',
                'home_defensive_actions': 'defensive_actions'}),
    matches_df[['match_date', 'away_team', 'away_tackles', 'away_interceptions', 
                'away_recoveries', 'away_defensive_actions']].rename(
        columns={'away_team': 'team', 'away_tackles': 'tackles', 
                'away_interceptions': 'interceptions', 'away_recoveries': 'recoveries',
                'away_defensive_actions': 'defensive_actions'})
]).sort_values(['team', 'match_date'])

# Calculate rolling averages for defensive metrics
defensive_data['tackles_rolling_avg_5'] = defensive_data.groupby('team')['tackles'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

defensive_data['interceptions_rolling_avg_5'] = defensive_data.groupby('team')['interceptions'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

defensive_data['defensive_actions_rolling_avg_5'] = defensive_data.groupby('team')['defensive_actions'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

print("\nRolling defensive averages calculated!")
print(defensive_data[defensive_data['team'] == 'Liverpool'].head(10))

In [ ]:
# Analyze defensive styles
team_defensive_style = defensive_data.groupby('team').agg({
    'tackles': 'mean',
    'interceptions': 'mean',
    'recoveries': 'mean',
    'defensive_actions': 'mean'
}).round(2)

team_defensive_style = team_defensive_style.sort_values('defensive_actions', ascending=False)

print("Team Defensive Intensity (avg per match):")
print(team_defensive_style)

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(team_defensive_style))

ax.barh(y_pos, team_defensive_style['defensive_actions'], color='steelblue')
ax.set_yticks(y_pos)
ax.set_yticklabels(team_defensive_style.index)
ax.set_xlabel('Average Defensive Actions per Match', fontsize=12)
ax.set_title('Team Defensive Intensity', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nHigh defensive actions = Aggressive, pressing style")
print("Low defensive actions = Passive, sit-deep style")

### Interpretation

A team that averages **20 tackles per game** is very different from one that averages **10**:

- **High-tackling team**: Probably pressing aggressively, trying to win the ball high up the pitch
- **Low-tackling team**: Might be playing a more passive, sit-deep-and-absorb-pressure style

Neither is inherently better—it depends on the opponent and the game situation!

---

## Pressing Intensity: PPDA

One of the most sophisticated defensive metrics is **Passes Allowed Per Defensive Action (PPDA)**.

This measures how many passes a team allows the opponent to make before attempting a defensive action (tackle, interception, or challenge).

- **Low PPDA** = Aggressive pressing (not letting opponent string passes together)
- **High PPDA** = Sitting back, allowing opponent to have the ball

### Formula:

$$\text{PPDA} = \frac{\text{Opponent Passes}}{\text{Defensive Actions}}$$

In [ ]:
# Calculate PPDA for home teams
matches_df['home_ppda'] = matches_df['home_opponent_passes'] / (matches_df['home_tackles'] + 
                                                                 matches_df['home_interceptions'])

# Calculate PPDA for away teams
matches_df['away_ppda'] = matches_df['away_opponent_passes'] / (matches_df['away_tackles'] + 
                                                                 matches_df['away_interceptions'])

print("PPDA calculated!")
print("\nSample:")
print(matches_df[['home_team', 'away_team', 'home_ppda', 'away_ppda']].head(10))

In [ ]:
# Create long-form for PPDA analysis
ppda_data = pd.concat([
    matches_df[['match_date', 'home_team', 'home_ppda']].rename(
        columns={'home_team': 'team', 'home_ppda': 'ppda'}),
    matches_df[['match_date', 'away_team', 'away_ppda']].rename(
        columns={'away_team': 'team', 'away_ppda': 'ppda'})
]).sort_values(['team', 'match_date'])

# Rolling average PPDA
ppda_data['ppda_rolling_avg_5'] = ppda_data.groupby('team')['ppda'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

# Analyze team pressing styles
team_pressing = ppda_data.groupby('team')['ppda'].mean().sort_values()

print("Team Pressing Intensity (PPDA):")
print(team_pressing)
print("\nLower PPDA = More aggressive pressing")
print("Higher PPDA = More passive, sitting back")

In [ ]:
# Visualize pressing styles
fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(team_pressing))

colors = ['darkgreen' if ppda < team_pressing.median() else 'darkred' 
          for ppda in team_pressing.values]

ax.barh(y_pos, team_pressing.values, color=colors, alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(team_pressing.index)
ax.set_xlabel('PPDA (Passes Per Defensive Action)', fontsize=12)
ax.set_title('Team Pressing Intensity', fontsize=14, fontweight='bold')
ax.axvline(team_pressing.median(), color='black', linestyle='--', linewidth=2, label='Median')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nGreen = High-pressing teams (low PPDA)")
print("Red = Low-pressing teams (high PPDA)")

### Real-World Examples

- **Liverpool under Klopp**: PPDA around 7-9 (very aggressive pressing)
- **Man City under Guardiola**: PPDA around 8-10 (high pressing)
- **Burnley**: PPDA around 15-18 (sit deep, absorb pressure)

PPDA is one of the best metrics for quantifying pressing intensity!

---

## Set Pieces: The Hidden Game

Set pieces (corners, free kicks) are often overlooked but can be decisive. Some teams are particularly dangerous from set pieces, while others struggle to defend them.

**Key insight**: Set pieces are more predictable than open play, so teams can practice and perfect specific routines.

In [ ]:
# Analyze set piece frequency
set_piece_data = pd.concat([
    matches_df[['match_date', 'home_team', 'home_corners', 'home_free_kicks']].rename(
        columns={'home_team': 'team', 'home_corners': 'corners', 'home_free_kicks': 'free_kicks'}),
    matches_df[['match_date', 'away_team', 'away_corners', 'away_free_kicks']].rename(
        columns={'away_team': 'team', 'away_corners': 'corners', 'away_free_kicks': 'free_kicks'})
]).sort_values(['team', 'match_date'])

# Rolling averages for set pieces
set_piece_data['corners_rolling_avg_5'] = set_piece_data.groupby('team')['corners'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

set_piece_data['free_kicks_rolling_avg_5'] = set_piece_data.groupby('team')['free_kicks'].transform(
    lambda x: x.rolling(window=5, min_periods=1).mean().shift(1)
)

print("Set piece features calculated!")
print("\nSample:")
print(set_piece_data[set_piece_data['team'] == 'Arsenal'].head(10))

In [ ]:
# Analyze team set piece frequency
team_set_pieces = set_piece_data.groupby('team').agg({
    'corners': 'mean',
    'free_kicks': 'mean'
}).round(2)

team_set_pieces = team_set_pieces.sort_values('corners', ascending=False)

print("Average Set Pieces per Match:")
print(team_set_pieces)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Corners
axes[0].barh(range(len(team_set_pieces)), team_set_pieces['corners'], color='coral')
axes[0].set_yticks(range(len(team_set_pieces)))
axes[0].set_yticklabels(team_set_pieces.index)
axes[0].set_xlabel('Avg Corners per Match', fontsize=12)
axes[0].set_title('Corners Won', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Free kicks
team_fk_sorted = team_set_pieces.sort_values('free_kicks', ascending=True)
axes[1].barh(range(len(team_fk_sorted)), team_fk_sorted['free_kicks'], color='skyblue')
axes[1].set_yticks(range(len(team_fk_sorted)))
axes[1].set_yticklabels(team_fk_sorted.index)
axes[1].set_xlabel('Avg Free Kicks per Match', fontsize=12)
axes[1].set_title('Free Kicks Won', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

---

# Tactical and Contextual Features: The Bigger Picture

Soccer doesn't happen in a vacuum. Context matters:
- How many days rest did the team have?
- Are they in good form or struggling?
- Is this a crucial match or a "dead rubber"?

Let's build features that capture this context.

## Rest and Fixture Congestion

Fatigue is real. A team playing their third match in 7 days is not the same as a team with 10 days rest.

**Days since last match** is a simple but powerful feature.

In [ ]:
# Calculate days since last match for each team
def calculate_rest_days(df):
    """Calculate days since last match for each team."""
    rest_data = []
    
    for team in df['home_team'].unique():
        # Get all matches for this team (home and away)
        team_matches = pd.concat([
            df[df['home_team'] == team][['match_date', 'home_team']].rename(columns={'home_team': 'team'}),
            df[df['away_team'] == team][['match_date', 'away_team']].rename(columns={'away_team': 'team'})
        ]).sort_values('match_date')
        
        # Calculate days since last match
        team_matches['days_since_last_match'] = team_matches['match_date'].diff().dt.days
        team_matches['days_since_last_match'] = team_matches['days_since_last_match'].fillna(14)  # First match
        
        rest_data.append(team_matches)
    
    return pd.concat(rest_data, ignore_index=True)

rest_df = calculate_rest_days(matches_df)

print("Rest days calculated!")
print("\nSample:")
print(rest_df[rest_df['team'] == 'Liverpool'].head(15))

In [ ]:
# Analyze fixture congestion
print("Rest Days Distribution:")
print(rest_df['days_since_last_match'].describe())

# Visualize
plt.figure(figsize=(10, 6))
plt.hist(rest_df['days_since_last_match'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Days Since Last Match', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Fixture Congestion Analysis', fontsize=14, fontweight='bold')
plt.axvline(rest_df['days_since_last_match'].median(), color='red', 
            linestyle='--', linewidth=2, label=f'Median: {rest_df["days_since_last_match"].median():.1f} days')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n< 3 days = Very congested (potential fatigue)")
print("3-7 days = Normal schedule")
print("> 7 days = Well rested")

## Momentum: Streaks and Form

**Momentum** is the feeling that a team is on a roll (or in a slump). We can quantify this with:
- Win/loss streaks
- Points from last N matches
- Form indicators

In [ ]:
# Create result column
def get_result(row, team):
    """Get match result from team's perspective."""
    if row['home_team'] == team:
        if row['home_score'] > row['away_score']:
            return 'W'
        elif row['home_score'] < row['away_score']:
            return 'L'
        else:
            return 'D'
    else:  # away team
        if row['away_score'] > row['home_score']:
            return 'W'
        elif row['away_score'] < row['home_score']:
            return 'L'
        else:
            return 'D'

# Create results dataframe
results_data = []
for team in matches_df['home_team'].unique():
    team_matches = pd.concat([
        matches_df[matches_df['home_team'] == team][['match_date', 'home_team', 'away_team', 'home_score', 'away_score']],
        matches_df[matches_df['away_team'] == team][['match_date', 'home_team', 'away_team', 'home_score', 'away_score']]
    ]).sort_values('match_date')
    
    team_matches['team'] = team
    team_matches['result'] = team_matches.apply(lambda row: get_result(row, team), axis=1)
    
    results_data.append(team_matches[['match_date', 'team', 'result']])

results_df = pd.concat(results_data, ignore_index=True).sort_values(['team', 'match_date'])

print("Results calculated!")
print("\nSample:")
print(results_df[results_df['team'] == 'Man City'].head(15))

In [ ]:
# Calculate win/loss streaks
def calculate_streak(results):
    """Calculate current streak (positive for wins, negative for losses)."""
    if len(results) == 0:
        return 0
    
    current = results.iloc[-1]
    streak = 0
    
    for i in range(len(results) - 1, -1, -1):
        if results.iloc[i] == current and current != 'D':
            if current == 'W':
                streak += 1
            else:  # Loss
                streak -= 1
        else:
            break
    
    return streak

results_df['streak'] = results_df.groupby('team')['result'].transform(
    lambda x: x.rolling(window=len(x), min_periods=1).apply(lambda y: calculate_streak(y), raw=False)
)

print("Streaks calculated!")
print("\nCurrent streaks:")
current_streaks = results_df.groupby('team')['streak'].last().sort_values(ascending=False)
print(current_streaks)

In [ ]:
# Calculate points from last 5 matches (3 for win, 1 for draw, 0 for loss)
def result_to_points(result):
    if result == 'W':
        return 3
    elif result == 'D':
        return 1
    else:
        return 0

results_df['points'] = results_df['result'].apply(result_to_points)

results_df['points_last_5'] = results_df.groupby('team')['points'].transform(
    lambda x: x.rolling(window=5, min_periods=1).sum().shift(1)
)

print("Form (points from last 5 matches):")
form_table = results_df.groupby('team')['points_last_5'].last().sort_values(ascending=False)
print(form_table)
print("\nMax possible: 15 points (5 wins)")

In [ ]:
# Visualize current form
fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(form_table))

colors = ['darkgreen' if pts >= 10 else 'orange' if pts >= 6 else 'darkred' 
          for pts in form_table.values]

ax.barh(y_pos, form_table.values, color=colors, alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(form_table.index)
ax.set_xlabel('Points from Last 5 Matches', fontsize=12)
ax.set_title('Current Team Form', fontsize=14, fontweight='bold')
ax.axvline(10, color='green', linestyle='--', alpha=0.5, label='Good Form (10+ pts)')
ax.axvline(6, color='orange', linestyle='--', alpha=0.5, label='Average Form (6+ pts)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Summary

In this notebook, you learned:

✅ **Defensive metrics** - tackles, interceptions, recoveries

✅ **PPDA (Passes Per Defensive Action)** - quantifying pressing intensity

✅ **Set piece features** - corners and free kicks

✅ **Rest days** - measuring fixture congestion and fatigue

✅ **Momentum features** - streaks and recent form

### Key Insights

**1. Defense is as important as offense**
- Different defensive styles (pressing vs sitting deep)
- PPDA is one of the best metrics for pressing intensity

**2. Context matters**
- Rest days affect performance
- Momentum and form are real

**3. Set pieces are significant**
- Can account for 30-40% of goals
- Some teams specialize in set pieces

**4. Multiple perspectives**
- Combine offensive and defensive features
- Use contextual features for better predictions

## Practice Exercises

1. **Goals conceded rolling average**: Create defensive form features.

2. **Clean sheets**: Track rolling count of matches without conceding.

3. **Defensive efficiency**: Calculate (goals conceded / defensive actions).

4. **Set piece goals**: If you have goal data, calculate set piece conversion rate.

5. **Fatigue indicator**: Create a feature for matches in last 7/14/21 days.

6. **Form momentum**: Calculate acceleration of form (is form improving or declining?).

7. **Head-to-head streaks**: Track streaks against specific opponents.

8. **Combined pressing**: Create a composite metric combining PPDA and defensive actions.